<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/microscopy-Core-ISMMS/ImageAnalysisCourse/blob/2026-workshop/notebooks/05_desktop_gui_complements.ipynb)

*Click the badge to open this notebook in Google Colab. For best performance, switch to a GPU runtime: Runtime → Change runtime type → T4 GPU.*

# Notebook 05 — Desktop GUI Complements

**Trainable Weka · QuPath Pixel Classifier · deepImageJ · StarDist (Fiji & QuPath)**

A walkthrough of five desktop-GUI workflows that complement the Colab labs. Unlike Notebooks 00–04, this notebook **is not executable**. It is a written + screenshot guide you read on workshop day and follow at your own laptop afterward.

---

## Why this notebook exists

The workshop's hands-on labs are Colab-first by design — zero install friction, identical environment for every attendee. That choice is the right one for the day, but it has a cost: three things that matter for real practice don't fit inside a Colab notebook.

1. **The trainable-segmenter loop without writing training code.** None of the Colab labs train a custom segmenter end-to-end. You run pretrained Cellpose-SAM, you prompt SAM, you watch CARE/fnet/pix2pix train on toy data — but you never paint your own labels and watch a model learn from them. *Trainable Weka Segmentation* (Fiji) and the *QuPath Pixel Classifier* close this gap with a brush tool and no Python.

2. **Whole-slide pathology images.** A typical H&E slide is gigapixels. It does not fit in Colab memory; it does not stream well over a notebook UI. *QuPath* is the dominant open-source desktop tool for these images and is essential to the broader pathology track in the curriculum.

3. **Pretrained DL models inside the imaging tool you already use.** Many lab members will never run Python. *deepImageJ* lets them apply BioImage Model Zoo models from inside Fiji; *StarDist* and *μSAM* have Fiji and QuPath surfaces. The same model — different surface for a different audience.

This notebook is the workshop's deliberate desktop chapter. Read it on workshop day; install Fiji and QuPath at your own pace afterward.

## What you will need

You do not need any of this to read along on workshop day. You need it to *do* the walkthroughs at home.

### Fiji
[Download Fiji](https://fiji.sc/) (≈ 400 MB; runs on macOS, Windows, Linux). Fiji is "ImageJ + a curated plugin distribution" — the standard tool in microscopy image analysis.

After installing, open Fiji and update via *Help → Update…* before the first session. The update mechanism also lets you subscribe to *update sites* — separate plugin channels — which is how you install StarDist, deepImageJ, and CSBDeep below.

### QuPath
[Download QuPath](https://qupath.github.io/) (≈ 200 MB; same platforms). QuPath is the standard open-source tool for whole-slide pathology and is also useful for tiled fluorescence. Built-in pixel and object classifiers, scriptable via Groovy.

### Sample data
Most exercises in this notebook use either a built-in sample (Fiji's *File → Open Samples → …*; QuPath's bundled tutorial slides) or one of the public images we used in Notebook 01:

- **Fiji built-in** — *File → Open Samples → HeLa Cells* (RGB fluorescence) or *Blobs* (single-channel test pattern).
- **scikit-image canonical samples** — exported as TIFFs from Notebook 01: `cells3d` middle slice (DAPI + membrane), `human_mitosis`, `cell`. If you want these on disk, run the snippet in *Appendix — Exporting sample images* at the bottom of this notebook.
- **QuPath sample slides** — the [QuPath tutorial slides](https://qupath.readthedocs.io/en/stable/docs/intro/getting_started.html#sample-data) include a small H&E and IHC slide that fit on a laptop.

### Tool versions targeted in this guide
- Fiji: **2.16+** (any recent 2025 build)  ⚠ *menu paths verified against Fiji 2.16; older builds may differ*
- QuPath: **0.5+**  ⚠ *menu paths verified against QuPath 0.5; the extension API changed in 0.4 → 0.5*

If you are on an older Fiji or QuPath, the workflow concepts are identical but the menu wording may not match exactly.

## A. Trainable Weka Segmentation in Fiji — the no-code trainable segmenter

This is the section that most directly fills the "you never trained a segmenter" gap from the labs. Trainable Weka Segmentation (TWS) is a built-in Fiji plugin: you paint pixel labels with a brush, click *Train*, and a random-forest classifier learns to segment your image. Five minutes from "open image" to "first prediction." Iterate by painting more labels where the model is wrong.

**Citation.** Arganda-Carreras et al., *Bioinformatics* 33(15):2424–2426, 2017. Open source.

**When to reach for it.**
- You have unlabeled images and need pixel-class segmentation (foreground/background, or N tissue classes).
- The pretrained models (Cellpose-SAM, StarDist, μSAM) are out-of-distribution for your sample type.
- You want to demonstrate the full *label → train → predict → iterate* loop to a colleague who does not code.
- The trainable model needs to live as a portable artifact (Weka exports a `.model` file you can re-apply to thousands of images headlessly).

**When not to reach for it.**
- Your images are gigapixel WSI — use QuPath's pixel classifier (Section B) instead.
- You need instance segmentation (separating touching objects) — TWS produces semantic segmentation; you'll need a downstream watershed or another instance step.
- You have plenty of paired labeled training data already — train a deep model (Cellpose fine-tune, StarDist, U-Net) for a higher ceiling.

### A.1 Open and prepare an image

1. Launch Fiji.
2. *File → Open Samples → HeLa Cells*  — or open one of the sample TIFFs from the Appendix.   ⚠ *Confirm exact submenu name on your Fiji build.*
3. If the image is multi-channel and you only want one channel for the demo, *Image → Color → Split Channels*, then close all but the channel you want to segment.
4. Convert to 8-bit if needed: *Image → Type → 8-bit*. TWS works best on 8-bit grayscale to start; you can layer extra channels in via the *Settings* dialog later.

> **[Screenshot 1]** *HeLa Cells sample image opened in Fiji, single channel selected, showing the standard ImageJ window (image canvas + tool palette + status bar).*

### A.2 Launch the Trainable Weka Segmentation plugin

*Plugins → Segmentation → Trainable Weka Segmentation*.   ⚠ *In some Fiji builds the entry is under Plugins → Segmentation → Trainable Weka Segmentation; in others under a sub-folder. Search via Help → Find Commands… and type "Weka" if you can't find it.*

The plugin opens its own window with: the image canvas in the center, two default classes ("class 1" and "class 2") in the right panel, and the toolbar across the top with *Train classifier*, *Get probability*, *Create result*, *Save classifier*, *Settings*, and others.

> **[Screenshot 2]** *TWS main window with the HeLa Cells image loaded, two default empty classes in the right panel, and the toolbar visible at the top.*

### A.3 Add and label classes

1. Rename "class 1" to **`cell`** and "class 2" to **`background`**. Click on the class label to rename.
2. *(Optional)* Add a third class **`nucleus`** via *Add new class*. This is where TWS earns its keep — multi-class semantic segmentation is one click.
3. Select the **freehand line** or **brush** tool from Fiji's main tool palette. The brush radius can be set with *Edit → Options → Line Width*.
4. With the `cell` class selected (click its name to make it active — it should highlight), draw inside several cells. You do not need to be perfect; a handful of strokes per class is enough to start. The plugin records the pixel coordinates of every stroke as labeled training pixels.
5. Click `background` and draw on several background regions.
6. *(If you added nucleus)* Click `nucleus` and draw inside a few nuclei.

> **[Screenshot 3]** *TWS window after labeling: brush traces visible inside three cells (red), in the background (green), and inside two nuclei (blue), with the class panel on the right showing the three named classes.*

### A.4 Train and inspect the classifier

1. Click **Train classifier**. The status bar shows feature extraction (Gaussian blur, Hessian, membrane projections, …) followed by random-forest training. On a laptop with a 512×512 image and ~50 brush strokes this takes 5–30 seconds.
2. The classifier output appears as a translucent overlay on the image — each pixel painted with the class color most likely under the model. Toggle the overlay with the **Toggle overlay** button to compare against the raw image.
3. Click **Get probability** to view per-class probability maps. This is the right way to inspect *uncertainty* — areas where the classifier hedges (probabilities near 0.5) are exactly where you want to add more brush strokes next.

> **[Screenshot 4]** *TWS overlay after first Train click: red/green/blue regions covering the image. Cells are mostly red, background mostly green, but several boundary pixels are clearly mis-classified — a deliberate teaching moment.*

> **[Screenshot 5]** *The probability map output (one image per class) opened as a stack, showing the cell-class probabilities ranging from 0 to 1 across the image.*

### A.5 Iterate

The first training pass will be wrong somewhere. That is not a failure — it is the *point*. Find a region where the overlay disagrees with what you see, paint a few more brush strokes of the correct class on top of the error, then click **Train classifier** again. The model retrains in seconds.

A useful loop:
- *Train* → toggle overlay off → look for misclassified pixels → toggle overlay on → paint corrections → *Train*.
- Three or four iterations usually gets you from "obviously wrong" to "good enough for downstream metrics."

**Reflection.** This is the supervised-learning loop in miniature. The brush strokes are your labeled training set; clicking *Train* fits a random forest on hand-engineered image features (Gaussian blur, Sobel, membrane projections, …) at multiple scales; the overlay is inference on the held-out pixels you did not label. Everything you learned about train/val/test in the lecture and Notebook 02 applies. **The metrics from Notebook 02 (IoU, Dice, count error) are the right way to validate a TWS output, on a held-out test image.**

> **[Screenshot 6]** *TWS window after three iterations of refinement, showing a clean overlay where cells, nuclei, and background are correctly separated.*

### A.6 Apply, export, save the classifier

1. **Create result** produces a single labeled image (an 8-bit indexed image with one value per class) that you can save with *File → Save As → TIFF*. This is your final segmentation.
2. **Save classifier** writes a `.model` file. You can reload this on a different image with *Load classifier* — that is the GUI equivalent of "trained model artifact" we talked about in the lecture. Re-apply with **Apply classifier** on a new image stack to do batch inference.
3. For headless / scripted batch processing, TWS exposes a Beanshell / Groovy / Python (Jython) scripting API. Search the [Fiji TWS documentation](https://imagej.net/plugins/tws/) for `runWekaSegmentation`.

> **[Screenshot 7]** *The "Create result" output: single-channel labeled image with three intensity levels (background, cell, nucleus), ready for export.*

### A.7 Going further — Labkit

[Labkit](https://imagej.net/plugins/labkit/) is the modern successor to TWS. Same paint-and-train loop, but the labeling tools are faster (better undo/redo, SAM-assisted polygon labeling), it scales to larger images, and it has an optional deep-learning backend (U-Net training via [BigDataViewer](https://imagej.net/plugins/bdv/)). Worth trying once you are comfortable with TWS — same mental model, smoother UX.

**When TWS is enough.** Random forests on hand-engineered features still beat tiny CNNs on small datasets, and TWS results are fast and interpretable. Don't reach for Labkit / DL until you've confirmed TWS isn't enough.

## B. QuPath Pixel Classifier — the trainable segmenter for whole-slide images

QuPath ships with a built-in pixel classifier that is the pathology-track equivalent of Trainable Weka. Same brush-and-train loop, but designed for whole-slide images: tiled inference, multi-resolution annotation, and integration with QuPath's object-detection and quantification pipeline.

**Citation.** Bankhead et al., *Scientific Reports* 7:16878, 2017 (the QuPath paper). The pixel-classifier feature is documented in the [QuPath manual, "Pixel classification" chapter](https://qupath.readthedocs.io/en/stable/docs/tutorials/pixel_classification.html).

**Why it earns its own section.** A typical H&E whole-slide image is 50,000 × 50,000 pixels at 40× magnification. You cannot open one in Fiji without tiling it yourself. QuPath does the tiling for you, runs the classifier at the resolution you choose, and gives you a tissue-class segmentation that is downstream-ready for cell detection and quantification. **For pathology, this is the canonical no-code workflow.**

### B.1 Open a slide and create a project

1. Launch QuPath.
2. *File → Project → Create project* — pick an empty folder. Projects are how QuPath manages multiple slides + their annotations + classifiers; you almost always want one.
3. *Add images* → drag in a slide from the QuPath tutorial slides or one of your own. QuPath reads many WSI formats via Bio-Formats.
4. Open the slide by double-clicking it in the project pane.

> **[Screenshot 8]** *QuPath main window with a tutorial H&E slide loaded, showing the slide canvas, project pane on the left, and overview thumbnail in the corner.*

### B.2 Annotate tissue classes

1. *Annotate → Set class* — define the classes you want to segment. For a basic tissue map this might be `tumor`, `stroma`, `necrosis`, `background`. The class list is shared across the project, so you only set it up once.
2. Pick the **brush** or **wand** tool from QuPath's annotation toolbar. Brush is fixed-radius; wand is intensity-based and follows tissue boundaries.
3. With your first class selected, draw on representative regions. As with TWS, a handful of strokes per class on a few different fields-of-view is enough to start.
4. Switch to the next class, repeat.

> **[Screenshot 9]** *The slide with tumor annotations (red), stroma (green), and background (blue) painted across several fields of view.*

### B.3 Train and preview live

QuPath's pixel classifier trains *live* — there is no explicit "train" button. You configure the classifier, and as you add or remove brush strokes the prediction updates.

1. *Classify → Pixel classification → Train pixel classifier*  ⚠ *menu wording verified against QuPath 0.5; older builds use "Create classifier" or similar.*
2. The training dialog shows: *Classifier* (random-forest is the default; ANN and other options exist), *Resolution* (which downsample level to compute features at — start at "Moderate"), *Features* (which image features to extract; start with the default set), *Output* (which class to display).
3. Click **Live prediction**. The slide canvas now shows the classifier's prediction as a translucent overlay, updating as you paint.
4. Iterate. Paint corrections; the live overlay updates within a second or two on most laptops.

> **[Screenshot 10]** *Pixel-classifier dialog open with Live prediction enabled, the slide overlay showing tissue class predictions across the slide.*

### B.4 Apply and export

1. **Save** the classifier (give it a name; it lives inside the project).
2. **Create objects** — turns the pixel prediction into geometric annotations (polygons) at a chosen minimum size. These objects are how QuPath does downstream quantification (area per class, density, …).
3. **Measure** — *Measure → Show annotation measurements* gives you per-class areas, fractions, and any quantitative measurements you've configured.
4. For batch processing across all slides in the project, *Run for project* applies the classifier to every image at once.

> **[Screenshot 11]** *Slide after Create objects: clean polygon annotations matching the tumor / stroma / background regions.*

### B.5 What this enables

This single classifier is the input to the rest of QuPath's pathology pipeline: cell detection inside *tumor* regions, quantification of marker positivity (IHC), spatial statistics across regions. The pixel classifier is the *foundation*; everything downstream is built on it.

**Cross-reference.** The validation framing from Notebook 02 — IoU per class, area-fraction error against ground truth on a held-out slide — applies identically here. The metrics do not care that the segmentation came from a GUI.

### B.6 Object classifier — train a downstream classifier on top

QuPath's *object classifier* is the second half of the no-code loop. After cell detection, you assign training labels to a few cells of each type (e.g., "lymphocyte", "tumor cell", "fibroblast") and QuPath trains a random forest on per-cell measurements. Same paint-and-train idiom, applied to objects rather than pixels.

This is beyond the scope of this notebook but follows directly from B.4 — covered in the [QuPath object classification tutorial](https://qupath.readthedocs.io/en/stable/docs/tutorials/cell_classification.html).

## C. deepImageJ — BioImage Model Zoo on the desktop

[deepImageJ](https://deepimagej.github.io/) is the Fiji surface for the [BioImage Model Zoo](https://bioimage.io). Anything you browsed via `bioimageio.core` in Notebook 04 — the structured DL model registry — you can run here without writing a line of Python.

**Citation.** Gómez-de-Mariscal et al., *Nature Methods* 18(10):1192–1195, 2021. ⚠ *Verified Notebook 04 references the same paper.*

**Why this matters for the workshop.** Notebook 04's BiMZ section was a Python API browse. Many lab members will never run that browse. deepImageJ gives them the same registry, accessed through the menu of the imaging tool they already use.

### C.1 Install via Fiji updater

1. *Help → Update…* opens the Fiji updater.
2. *Manage update sites* in the lower-left.
3. Tick **deepImageJ** in the list.   ⚠ *If you don't see it, click "Add update site" and enter `https://sites.imagej.net/deepImageJ/` — verify URL on the deepImageJ docs page before public release.*
4. Close the update-sites dialog and click **Apply changes**. Restart Fiji.

After restart you should have a *Plugins → deepImageJ* submenu.

> **[Screenshot 12]** *Fiji updater with the deepImageJ update site checked, after a successful update.*

### C.2 Browse and load a BiMZ model

1. *Plugins → deepImageJ → deepImageJ Run*. ⚠ *Menu wording may differ on the version you install.*
2. The dialog lists models available locally. Click **Add model** to fetch from the BioImage Model Zoo. The browser shows model name, task tag (segmentation / restoration / …), and citation.
3. Pick a model whose task matches your image. Good first choices: a 2D fluorescence segmentation model for the HeLa Cells sample, or a denoising model for a noisy fluorescence image.
4. The download is a few hundred megabytes per model.

> **[Screenshot 13]** *deepImageJ Run dialog open with a list of available BiMZ models, one selected with its citation and input/output specs visible.*

### C.3 Run on an image

1. Open the image in Fiji first.
2. *Plugins → deepImageJ → deepImageJ Run* → select the model → **Run**. The plugin tiles the image automatically if it is bigger than the model's expected input shape.
3. The output is a new image (or stack) with the prediction. Save with *File → Save As → TIFF*.

> **[Screenshot 14]** *Side-by-side: input image and deepImageJ-Run output for a segmentation model, with the model's citation visible in the dialog.*

### C.4 What this enables

The same model from the registry runs in three places now: Notebook 04 via `bioimageio.core` (Python), deepImageJ (Fiji desktop), and several QuPath extensions (next sections). **Same model, same outputs (within numerical precision), three audiences.**

⚠ **BiMZ spec compatibility.** The BioImage Model Zoo's `rdf.yaml` model description spec has gone through several versions (0.4 → 0.5). Older deepImageJ builds may not load v0.5 models; newer ones may not load v0.3 models. If a model fails to load, check the deepImageJ release notes for spec compatibility before assuming the model is broken.

## D. StarDist Fiji plugin — pretrained nuclear segmentation, one click

StarDist is one of the workhorse pretrained models for fluorescence-microscopy nuclear segmentation. It predicts star-convex polygons rather than free-form masks, which is a strong inductive bias for nucleus shapes and gives clean instance segmentation out of the box.

**Citation.** Schmidt, Weigert, Broaddus, Myers, *MICCAI 2018*. ⚠ *Verified Notebook 04 catalogs the same paper.*

**Why include it here.** Notebook 04 catalogs StarDist as a ZeroCostDL4Mic Colab. The Fiji plugin is the same pretrained model, accessible from the menu of Fiji. For nuclei in fluorescence images, this is often the fastest path from "open image" to "instance-segmented nuclei."

### D.1 Install via update site

1. *Help → Update…* → *Manage update sites*.
2. Tick **CSBDeep** and **StarDist**.   ⚠ *Verify both update-site names on the StarDist Fiji docs before public release.*
3. **Apply changes** → restart.

After restart you should have a *Plugins → StarDist* submenu.

### D.2 Run on an image

1. Open a fluorescence image with nuclei (e.g., DAPI channel of HeLa Cells, or `human_mitosis` from the sample data).
2. *Plugins → StarDist → StarDist 2D*.   ⚠ *Exact submenu wording.*
3. The dialog lets you pick a pretrained model. Two safe defaults: `Versatile (fluorescent nuclei)` and `Versatile (H&E nuclei)`. Pick the one that matches your modality.
4. Run. The output is an instance-labeled image (one integer per nucleus) plus an ROI manager populated with one ROI per detected nucleus.

> **[Screenshot 15]** *StarDist 2D dialog with the pretrained model chosen, the source image visible, and the segmentation output overlay showing instance-segmented nuclei.*

### D.3 Where this fits the workshop

- **Validation.** The Notebook 02 metrics apply directly. Compute IoU, Dice, count error against a hand-labeled ground-truth ROI set.
- **When to pick StarDist over Cellpose-SAM.** StarDist is faster on nuclei specifically and gives cleaner star-convex shapes; Cellpose-SAM generalizes more broadly across morphologies. Try both on your data; the lecture's "show failures" framing applies — pick whichever fails *less* on your samples.
- **The 3D variant.** StarDist also has a 3D model, exposed as *Plugins → StarDist → StarDist 3D*. Use it on confocal stacks; the inductive bias still helps.

## E. QuPath StarDist extension — H&E nuclear segmentation at WSI scale

The QuPath StarDist extension brings the same StarDist models into QuPath, where they run *across whole-slide images* with proper tiling. For pathology, the H&E nucleus model is the canonical first step in any cell-based quantification pipeline.

**Citation.** Same StarDist paper (Schmidt et al., 2018) plus the QuPath extension authored by Pete Bankhead's group. ⚠ *Verify the extension's current GitHub repo and maintainer before public release; QuPath's extension API changed at 0.4 → 0.5 and not all extensions track every release.*

**Why include it here.** A pathology slide has hundreds of thousands of nuclei. You cannot open it in Fiji and click *Plugins → StarDist*. QuPath's extension does the tiling, runs StarDist on each tile, and merges the results into proper QuPath cell objects ready for downstream quantification.

### E.1 Install the extension

1. Download the extension JAR from its GitHub releases page. ⚠ *Confirm the canonical repo at install time — historically `qupath/qupath-extension-stardist`, but verify before publishing this notebook.*
2. *Edit → Preferences → Extensions* in QuPath, or simply drag the JAR onto the QuPath window.
3. QuPath copies the JAR into its extensions folder and restarts.

After restart you should have an *Extensions → StarDist* menu, and the extension exposes a Groovy API for scripting.

> **[Screenshot 16]** *QuPath Preferences → Extensions panel showing the StarDist extension installed.*

### E.2 Run on a slide

The QuPath StarDist extension is most commonly invoked from a Groovy script rather than a dialog (this gives reproducibility and lets you batch across a project). A minimal H&E-nuclei script looks roughly like:

```groovy
// QuPath Groovy — illustrative skeleton; consult the extension's docs
// for the exact API current to your QuPath + extension versions.
import qupath.ext.stardist.StarDist2D

def stardist = StarDist2D.builder('he_heavy_augment.pb')   // a bundled model
    .threshold(0.5)
    .normalizePercentiles(1, 99)
    .pixelSize(0.5)
    .cellExpansion(5.0)
    .build()

def selected = getSelectedObjects()
stardist.detectObjects(getCurrentImageData(), selected)
```

⚠ *The exact builder method names and the bundled model filename change between extension releases. Treat the snippet above as a placeholder — copy from the [QuPath StarDist documentation](https://qupath.readthedocs.io/en/stable/docs/deep/stardist.html) at the time you install.*

> **[Screenshot 17]** *QuPath script editor with a StarDist H&E script loaded, executed across a slide region, and the resulting cell objects visible on the canvas.*

### E.3 Where this fits the workshop

This is the gateway exercise into the **pathology track** of the broader curriculum. The morphology metrics, validation framing, and stewardship questions from the lecture all apply at slide scale: a whole-slide segmentation that drops 5% of nuclei translates to thousands of missed cells per slide, and the per-class IoU on H&E is the right number to report alongside any biological claim.

For attendees on a pure microscopy track this section is optional. For anyone whose data is pathology, it is essential.

## How this notebook connects to the rest of the workshop

The five GUI workflows above are not a parallel curriculum — they are the desktop counterparts of things you already touched in the labs and the lecture. Cross-references:

- **Notebook 01 (Cellpose-SAM segmentation)**: same pretrained-model paradigm, different surface. The Cellpose authors maintain Fiji and napari plugins; the [Cellpose Fiji plugin](https://github.com/MouseLand/cellpose) is the GUI counterpart of Notebook 01. ⚠ *Plugin status changes; verify which Fiji entry-point is canonical at install time.*
- **Notebook 02 (validation)**: every metric in Notebook 02 — IoU, Dice, count error, area-fraction error — applies identically to the outputs of TWS, QuPath Pixel Classifier, deepImageJ, and StarDist. **The validation framework does not change because the model came from a GUI.** Bring a held-out test image with ground-truth labels, compute the same numbers.
- **Notebook 03a (denoising)**: the [CARE Fiji plugin](https://github.com/CSBDeep/CSBDeep_website) (CSBDeep update site) runs the same denoising model from inside Fiji. If you trained a CARE model in ZeroCostDL4Mic, you can deploy it in Fiji.
- **Notebook 03b (foundation models)**: the [μSAM plugin](https://github.com/computational-cell-analytics/micro-sam) has both a napari interface and a [QuPath extension](https://github.com/computational-cell-analytics/micro-sam) — the desktop counterparts of the foundation-model lab.
- **Notebook 04 (community platforms)**: deepImageJ (Section C above) is the Fiji surface for the same BioImage Model Zoo registry that Notebook 04 browsed via `bioimageio.core`.
- **Lecture 1**: the "responsible use & validation" pillar carries verbatim. A trainable Weka classifier without a held-out test set has the same problem as a CNN without a held-out test set. The "show failures" framing applies — try TWS or the QuPath Pixel Classifier on a deliberately out-of-distribution sample and watch it fail.
- **Lecture 2**: the bridge talk between the morning lecture and the labs is the natural home for a 10-minute live GUI demo. *Trainable Weka → deepImageJ paired demo* in Fiji is the highest-pedagogical-value content this notebook supports — see the workshop schedule.

## When to use which tool

A short decision aid for "which GUI for this task":

| You want… | Reach for | Why |
|---|---|---|
| To paint labels and train a segmenter on a single fluorescence image | **Trainable Weka** (Fiji) | Built-in, fast, interpretable, exports a portable classifier. |
| Same as above but on a whole-slide H&E or IHC | **QuPath Pixel Classifier** | WSI-aware tiling, integrates with QuPath quantification. |
| To run a community-maintained pretrained DL model from inside Fiji | **deepImageJ** | Direct surface for the BioImage Model Zoo registry. |
| Pretrained nucleus segmentation, one click, fluorescence | **StarDist Fiji plugin** | Star-convex shape prior, clean instance output. |
| Pretrained nucleus segmentation, one script, H&E whole slide | **QuPath StarDist extension** | Tiles the slide, merges into QuPath cell objects. |
| To label and train at WSI scale, including tracking | The other QuPath classifiers + (eventually) Labkit / DL backends | Active area; check the [QuPath docs](https://qupath.readthedocs.io/) for what's current. |

## What is **not** in this notebook (and where to look instead)

- **Cellpose human-in-the-loop fine-tuning.** Cellpose 2.0+ supports a GUI-driven fine-tune workflow inside the Cellpose desktop app. If you want a *deep-learning* trainable segmenter (rather than the random-forest TWS), this is the canonical path. Documented in the [Cellpose docs](https://cellpose.readthedocs.io/).
- **napari + plugins.** napari is the modern Python-native viewer with a deep plugin ecosystem ([napari-segment-anything], [napari-cellpose], [napari-cellseg3d], …). It bridges code-first and GUI workflows in a way Fiji cannot. A separate notebook is warranted.
- **Tracking GUIs.** TrackMate (Fiji) and ELEPHANT, Mastodon, btrack — out of scope for this workshop's segmentation focus.
- **Slide-management infrastructure.** OMERO, BIA Image Archive, MoBIE — adjacent to the analysis tools above but a different workflow concern.

## Screenshot capture checklist

The 17 placeholder boxes above need real screenshots before this notebook is published. Capture them in a single session against the tool versions noted above (Fiji 2.16+, QuPath 0.5+) and replace the `[Screenshot N: …]` markers in the source markdown.

| # | Tool | Where | What to capture |
|---|------|-------|-----------------|
| 1 | Fiji | Section A.1 | HeLa Cells sample image, single channel selected, full Fiji window. |
| 2 | Fiji TWS | Section A.2 | TWS main window, two empty default classes, toolbar visible. |
| 3 | Fiji TWS | Section A.3 | TWS window after labeling: brush traces visible for `cell` (red), `background` (green), `nucleus` (blue). Three-class panel on the right. |
| 4 | Fiji TWS | Section A.4 | First Train output overlay — deliberately imperfect, with visible boundary errors. |
| 5 | Fiji TWS | Section A.4 | Probability map output as a stack, one image per class, full intensity range. |
| 6 | Fiji TWS | Section A.5 | TWS overlay after 3 iterations of refinement — clean class separation. |
| 7 | Fiji TWS | Section A.6 | "Create result" output: 8-bit indexed image with one value per class. |
| 8 | QuPath | Section B.1 | Tutorial H&E slide loaded, project pane left, overview thumbnail. |
| 9 | QuPath | Section B.2 | Slide with `tumor` (red), `stroma` (green), `background` (blue) brush annotations. |
| 10 | QuPath | Section B.3 | Pixel-classifier dialog open, Live prediction enabled, overlay visible. |
| 11 | QuPath | Section B.4 | Slide after Create objects — clean polygons. |
| 12 | Fiji updater | Section C.1 | Updater dialog with deepImageJ update site checked. |
| 13 | Fiji deepImageJ | Section C.2 | deepImageJ Run dialog with BiMZ models listed, one selected. |
| 14 | Fiji deepImageJ | Section C.3 | Side-by-side: input image and deepImageJ output, with citation visible. |
| 15 | Fiji StarDist | Section D.2 | StarDist 2D dialog with pretrained model chosen, segmentation overlay. |
| 16 | QuPath | Section E.1 | Preferences → Extensions panel showing StarDist installed. |
| 17 | QuPath | Section E.2 | Script editor with StarDist H&E Groovy script, executed across a slide region, cell objects visible. |

**Suggested capture protocol:**

1. Run Fiji and QuPath on a stable laptop/desktop with a 1920×1200 (or larger) display so windows aren't cropped.
2. Use the same sample image across Sections A and D (HeLa Cells with a clear DAPI channel) for visual continuity.
3. For QuPath sections (B, E), use the tutorial H&E slide so anyone replicating can match.
4. PNG, full-window screenshots; crop after the fact rather than capturing partial windows.
5. After capture, save under `notebooks/figures/05/` and replace each `[Screenshot N: ...]` in the source markdown with a Markdown image link of the form `![Screenshot N — short title](figures/05/screenshot_N.png)`.

## Appendix — Exporting sample images for this notebook

If you want the same canonical samples Notebook 01 uses on disk for Fiji to open, run this snippet in any Python environment with `scikit-image` installed:

```python
from skimage import data
from skimage.io import imsave
import numpy as np
from pathlib import Path

out = Path("sample_data")
out.mkdir(exist_ok=True)

# 2D fluorescence — single nuclei field
imsave(out / "human_mitosis.tif", data.human_mitosis())

# 2D fluorescence — single cell
imsave(out / "cell.tif", data.cell())

# 3D confocal — middle slice
cells = data.cells3d()  # (z, c, y, x) = (60, 2, 256, 256)
mid = cells[cells.shape[0] // 2]   # (c, y, x)
imsave(out / "cells3d_mid_dapi.tif", mid[0])
imsave(out / "cells3d_mid_membrane.tif", mid[1])

print("Wrote sample images to:", out.resolve())
```

Drop the resulting TIFFs onto the Fiji or QuPath window to open.

## References

- Arganda-Carreras, I., Kaynig, V., Rueden, C., Eliceiri, K. W., Schindelin, J., Cardona, A., Sebastian Seung, H. **Trainable Weka Segmentation: a machine learning tool for microscopy pixel classification.** *Bioinformatics* 33(15):2424–2426, 2017. ✓
- Bankhead, P. et al. **QuPath: open source software for digital pathology image analysis.** *Scientific Reports* 7:16878, 2017. ✓
- Gómez-de-Mariscal, E. et al. **DeepImageJ: a user-friendly environment to run deep learning models in ImageJ.** *Nature Methods* 18(10):1192–1195, 2021. ⚠ *Citation cross-checked against Notebook 04; verify volume/issue at publication time.*
- Schmidt, U., Weigert, M., Broaddus, C., Myers, G. **Cell Detection with Star-convex Polygons.** *MICCAI* 2018. ✓
- Stringer, C., Wang, T., Michaelos, M., Pachitariu, M. **Cellpose: a generalist algorithm for cellular segmentation.** *Nature Methods* 18:100–106, 2021. ✓ (background reference for the Cellpose Fiji plugin pointer in Section C.)

For the corresponding Colab labs, see Notebooks 01–04 in the sidebar.

<!-- DATASET-AUDIT-PATCH -->
---
## Real microscopy datasets to explore next

The Fiji / QuPath workflows in this notebook expect a real image to load. Recommended sources for desktop hands-on:

- **BBBC005 ground truth** — [https://bbbc.broadinstitute.org/BBBC005](https://bbbc.broadinstitute.org/BBBC005) — 12 MB ZIP — pairs of in-focus images + binary masks. Ideal for trying TWS / Pixel Classifier.
- **BBBC020** — [https://bbbc.broadinstitute.org/BBBC020](https://bbbc.broadinstitute.org/BBBC020) — Multi-channel macrophages — good StarDist target.
- **Allen Cell Imaging Collections** — [https://www.allencell.org/data-downloading.html](https://www.allencell.org/data-downloading.html) — OME-TIFF hiPSC volumes — strong fit for deepImageJ / 3D Fiji.
- **Cell Image Library (per-image license)** — [https://www.cellimagelibrary.org/pages/datasets](https://www.cellimagelibrary.org/pages/datasets) — 12,000+ datasets at UCSD CRBS; filter by Public Domain or CC-BY before redistributing.
- **IDR** — [https://idr.openmicroscopy.org/](https://idr.openmicroscopy.org/) — OMERO sample images for QuPath WSI workflows.

Full audit and notebook ↔ dataset mapping in [`datasets_audit.md`](https://github.com/microscopy-Core-ISMMS/ImageAnalysisCourse/blob/2026-workshop/datasets_audit.md).
